<a href="https://colab.research.google.com/github/gokulbytes/personalized-news-recommendation-engine/blob/main/main_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [ ]:
# Install necessary libraries
!pip install pandas numpy scikit-learn -q

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# main

## behaviors_df

In [ ]:
# Load the behaviors dataset
behaviors_df = pd.read_csv('/content/behaviors.tsv', sep='\t', header=None, names=['UserID', 'TimeStamp', 'History', 'Impressions'])
# Display the first few rows of the dataframe
behaviors_df.head()

,UserID,TimeStamp,History,Impressions
1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [ ]:
# Display information about the dataframe
behaviors_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 156965 entries, 1 to 156965
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   UserID       156965 non-null  object
 1   TimeStamp    156965 non-null  object
 2   History      153727 non-null  object
 3   Impressions  156965 non-null  object
dtypes: object(4)
memory usage: 6.0+ MB


## news_df

In [ ]:
# Load the news dataset
news_df = pd.read_csv('/content/news.tsv', sep='\t', header=None, names=['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL','Title Entities','Abstract Entities'])
# Display the first few rows of the dataframe
news_df.head()

,NewsID,Category,SubCategory,Title,Abstract,URL,Title Entities,Abstract Entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [ ]:
# Display information about the dataframe
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   NewsID             51282 non-null  object
 1   Category           51282 non-null  object
 2   SubCategory        51282 non-null  object
 3   Title              51282 non-null  object
 4   Abstract           48616 non-null  object
 5   URL                51282 non-null  object
 6   Title Entities     51279 non-null  object
 7   Abstract Entities  51278 non-null  object
dtypes: object(8)
memory usage: 3.1+ MB


## Data Preprocessing

### behaviors_df

In [ ]:
# Remove duplicate rows and reset the index
behaviors_df = behaviors_df.drop_duplicates().reset_index(drop=True)

In [ ]:
# Select UserID and Impressions columns
user_impressions_df = behaviors_df[['UserID', 'Impressions']].copy()

### news_df

In [ ]:
# Select relevant columns from news_df
news_content_df = news_df[['NewsID', 'Category', 'SubCategory', 'Title','Abstract', 'URL']].copy()

In [ ]:
# Remove duplicate rows and reset the index
news_content_df = news_content_df.drop_duplicates().reset_index(drop=True)

In [ ]:
# Combine title, abstract, category, and subcategory into a single 'Content' column
news_content_df['Content'] = (news_content_df['Title'] + ' ' + news_content_df['Abstract'].fillna('') + ' ' + news_content_df['Category'] + ' ' + news_content_df['SubCategory'])

In [ ]:
# Drop the original columns
news_content_df = news_content_df.drop(columns=['Title', 'Abstract', 'Category', 'SubCategory'])

## Data Transformation

In [ ]:
# Split the Impressions string into a list of strings
user_impressions_df['Impressions'] = user_impressions_df['Impressions'].str.split()

In [ ]:
# Extract NewsIDs where the impression ended with '-1' (clicked)
user_impressions_df['Clicked'] = user_impressions_df['Impressions'].apply(lambda x:[i.split('-')[0] for i in x if i.endswith('-1')])

In [ ]:
# Drop the original Impressions column as it's no longer needed
user_impressions_df.drop(columns = ['Impressions'], inplace=True)

In [ ]:
# Rename the 'Clicked' column to 'NewsID' for clarity
user_impressions_df = user_impressions_df.rename(columns={'Clicked' : 'NewsID'})

In [ ]:
# Explode the NewsID column so each NewsID in the list gets its own row
user_impressions_df_exploded = user_impressions_df.explode('NewsID').reset_index(drop=True)

In [ ]:
# Remove duplicate UserID and NewsID combinations
user_impressions_df_exploded = user_impressions_df_exploded.drop_duplicates().reset_index(drop=True)

### final_df

In [ ]:
# Merge the exploded user impressions with the news content data based on NewsID
final_df = pd.merge(user_impressions_df_exploded, news_content_df, on='NewsID', how='left')
# Display the first few rows of the resulting dataframe
final_df.head()

,UserID,NewsID,URL,Content
0,U13740,N55689,https://assets.msn.com/labs/mind/BBWAPO6.html,"Charles Rogers, former Michigan State football..."
1,U91836,N17059,https://assets.msn.com/labs/mind/BBWE1bu.html,No. 1 milk company declares bankruptcy amid dr...
2,U73700,N23814,https://assets.msn.com/labs/mind/AAHr37p.html,I moved from the US to the UK. Here are the 8 ...
3,U34670,N49685,https://assets.msn.com/labs/mind/BBWyk8E.html,Broadway Star Laurel Griggs Suffered Asthma At...
4,U8125,N8400,https://assets.msn.com/labs/mind/BBOA5nc.html,These are the new cars that depreciate least D...


In [ ]:
final_df.shape # Display the shape of the final dataframe

(234468, 4)

### reduced_df

In [ ]:
# Reduce the dataframe for demonstration purposes and remove duplicates based on NewsID
reduced_df = final_df.head(10000).drop_duplicates(subset=['NewsID']).reset_index(drop=True)
reduced_df.shape # Display the shape of the reduced dataframe

(2196, 4)

## Feature Extraction

In [ ]:
# Initialize TfidfVectorizer and fit/transform the 'Content' column
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(reduced_df['Content'])

## Cosine Similarity

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

## Recommendation Function

In [ ]:
def get_recommendations(news_id, cosine_sim=cosine_sim, df=reduced_df):
    try:
        idx = df[df['NewsID'] == news_id].index[0]
    except IndexError:
        print(f"NewsID {news_id} not found in the dataframe.")
        return []

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    news_indices = [i[0] for i in sim_scores]
    recommended_news_ids = df['NewsID'].iloc[news_indices].tolist()
    return recommended_news_ids

### Testing

In [ ]:
# Get recommendations for a sample news ID and print the results
sample_news_id = reduced_df['NewsID'].iloc[0]
recommended_news = get_recommendations(sample_news_id)
print(f"Recommendations for {sample_news_id}: {recommended_news}")

Recommendations for N55689: ['N2476', 'N36638', 'N26649', 'N52193', 'N64513', 'N19477', 'N333', 'N44578', 'N40326', 'N13259']


## Save Model

In [ ]:
import joblib

joblib.dump(final_df, 'content_based_filtering.pkl')

print("Model saved successfully!")

Model saved successfully!
